In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
BRONZE_PATH = "abfss://bronze@pravdatalake.dfs.core.windows.net"
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"

In [0]:
category_df = spark.read.format("csv")\
    .option("header", True)\
    .option("inferSchema", True)\
    .load(f"{BRONZE_PATH}/netflix_category")

In [0]:
category_df.display()

In [0]:
silver_category = (
    category_df
    .withColumn("show_id", trim(col("show_id")))
    .withColumn("listed_in", trim(col("listed_in")))
    .withColumn(
        "listed_in",
        when(col("listed_in") == "", None)
         .otherwise(col("listed_in"))
    )
    .filter(col("show_id").isNotNull())
    .filter(col("listed_in").isNotNull())
    .dropDuplicates(["show_id", "listed_in"])
    .withColumn("silver_ingestion_timestamp", current_timestamp())
)



In [0]:
silver_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{SILVER_PATH}/netflix_category")

In [0]:
spark.sql("SHOW SCHEMAS IN netflix_catalog").display()